# 14 N-glycan Logistic Regression

## Purpose
This notebook asks a narrower, more controlled question than the full classifier notebooks:

- if we freeze one saved embedding model state
- and fit only a simple logistic-regression probe
- how well do different internal embedding layers separate `N-glycan` rows from other glycans?

This keeps the downstream model intentionally simple so the comparison emphasizes the representation itself rather than classifier capacity. In the cleaned version, the default story is how one pretrained MLM changes from input embeddings to middle-layer states to the final layer.


## Setup note

This notebook is closer to notebook `11b` than notebook `10`:

- code stays in GitHub
- prepared classification tables and checkpoints stay in Drive
- embeddings are recomputed from saved model folders
- the only trained model inside this notebook is a small sklearn logistic regression

The intended use is one tokenizer family plus one exact experiment, starting with the `pretrained_mlm` lineage, fixing one saved model folder such as `best_model`, and comparing ordered embedding layers inside that same model.


## Runtime setup

This cell mounts Google Drive, synchronizes the repository, and installs the lightweight packages needed for embedding extraction, the logistic-regression probe, and the shared UMAP projection.

**Expected output**
- confirmation that Drive is mounted
- confirmation that the repository was cloned or updated
- the active repository directory in Colab


In [ ]:
import os
import subprocess
import sys

from google.colab import drive

drive.mount('/content/drive')

# Keep tqdm in plain-text mode for cleaner notebook logs.
from tqdm.std import tqdm as plain_tqdm
import tqdm.auto as tqdm_auto
tqdm_auto.tqdm = plain_tqdm
try:
    import tqdm.notebook as tqdm_notebook
    tqdm_notebook.tqdm = plain_tqdm
except Exception:
    pass

!pip install -q transformers scikit-learn umap-learn

GITHUB_OWNER = "hb791-dev"
REPO_NAME = "glycan-roberta"
GITHUB_REF = "main"
REPO_URL = f"https://github.com/{GITHUB_OWNER}/{REPO_NAME}.git"
REPO_DIR = f"/content/{REPO_NAME}"

if not os.path.exists(REPO_DIR):
    print("Cloning repository...")
    subprocess.run(["git", "clone", "--quiet", REPO_URL, REPO_DIR], check=True)
else:
    print(f"Reusing existing repo at {REPO_DIR}")
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", GITHUB_REF], check=True)

%cd {REPO_DIR}

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print(f"Active repo directory: {REPO_DIR}")


## Import notebook helpers

This notebook keeps the repeatable mechanics in `src/classification_embedding_logreg.py` so the user-facing cells can stay focused on run selection and interpretation.

We run this cell early so the rest of the notebook can call a small set of readable helper functions instead of embedding the full logistic-regression workflow inline.

**Expected output**
- no printed output under normal conditions
- a standard Python import error only if the runtime setup step failed or a required package is missing

**How to interpret the result**
- if this cell runs quietly, the notebook is ready to build the run list and load the prepared classification tables
- if an import fails, fix the setup step before editing the later analysis cells


In [ ]:
import importlib
from pathlib import Path

import pandas as pd
from IPython.display import HTML, display

import src.classification_embedding_logreg as classification_embedding_logreg
importlib.reload(classification_embedding_logreg)

from src.classification_embedding_logreg import (
    CURRENT_PROJECT_LABEL_SOURCE,
    DEFAULT_STRUCTURAL_CONTRADICTION_POLICY,
    STRUCTURAL_RULE_LABEL_SOURCE,
    SUPPORTED_PROBE_LABEL_SOURCES,
    SUPPORTED_STRUCTURAL_CONTRADICTION_POLICIES,
    build_classifier_run_label_resolution_table,
    build_embedding_logreg_run_config,
    build_embedding_logreg_output_paths,
    build_layer_run_specs,
    build_registry_run_specs,
    build_run_manifest,
    export_public_embedding_logreg_html,
    load_run_registry,
    prepare_probe_dataframe_for_notebook14,
    resolve_classifier_run_label_choices,
    resolve_run_registry_path,
    run_embedding_logreg_suite,
    save_embedding_logreg_run_config,
)
from src.notebook_utils import (
    SUPPORTED_TOKENIZER_FAMILIES,
    build_responsive_image_html,
    validate_tokenizer_family,
)
from src.similarity import build_public_export_dir, build_public_report_subdir


## User settings

This is the main cell to edit before running the notebook.

**What to choose here**
- which tokenizer family and exact pretrained experiment to probe
- which comparison mode to run: one fixed model across layers, or one fixed layer across model variants
- which probe target to run: the original `N-glycan` binary task or the harder N-glycan subclass task
- which model lineage or model variants should be compared
- which one saved model folder from that lineage should be used, such as `best_model`
- which embedding layers from that fixed model should be compared in layer mode
- which one embedding layer should be fixed in variant mode
- which pooling rule should provide the embedding vectors
- which glycan accessions should be highlighted across the shared UMAP panels
- whether unlabeled rows should stay in the full corpus as non-`N-glycan` examples
- which N-glycan subclass buckets should be kept when the harder subclass probe is selected
- the logistic-regression and shared-UMAP settings
- whether to also copy a clean public HTML folder after the report is created

The notebook supports two related comparison layouts for one exact experiment. In `variant_fixed_layer` mode, it compares several model variants at one fixed embedding layer. In `layer_progression` mode, it holds one model fixed and compares several layers inside that model. It also supports two probe targets: the original binary `N-glycan` versus other task, and a harder N-glycan-only subclass probe across `High mannose`, `Complex`, and `Hybrid` rows.

**Expected output**
- this cell only defines notebook settings; it does not run the comparison yet

**How to interpret the result**
- fill `EXPERIMENT_NAME` with the exact pretrained run you want to probe
- use `COMPARISON_MODE = "variant_fixed_layer"` when you want to hold the embedding layer fixed and compare `pretrained_mlm`, `classification_mlm_init`, and `classification_random_init`
- use `COMPARISON_MODE = "layer_progression"` when you want to hold one saved model fixed and compare several internal layers
- keep `MODEL_SUBDIR = "best_model"` when you want to reuse the selected saved model folder across either mode
- set `FIXED_EMBEDDING_LAYER_INDEX = 0` when you want the input-embedding control across model variants
- set `EMBEDDING_LAYER_INDICES` to the ordered layers you want to compare in layer mode; for the default `L6` model, `0` is input embeddings, `3` is a middle encoder layer, and `-1` is the final layer
- keep `SPLITS_TO_INCLUDE = ("train", "val", "test")` if you want the shared UMAP to see the full probe corpus while the logistic regression still fits on `train` and reports on held-out `test`
- use `PROBE_TARGET_MODE = "n_glycan_subclass_multiclass"` when you want the harder N-glycan-only subclass probe instead of the easier binary task
- review the edge-case tables after preparation because rare multi-label glycans, mixed broad classes, or unsupported N-glycan subclass labels can be excluded from the harder probe
- adjust the classifier run-label candidate tuples only when one of the compared variants uses a fine-tuned lineage
- leave `PUBLIC_EXPORT_ENABLED = False` until the local Drive report looks correct, then turn it on to prepare the public-ready folder


In [ ]:
from pathlib import Path

# Update DRIVE_ROOT if the project folder uses a different Google Drive path.
DRIVE_ROOT = Path('/content/drive/MyDrive/ProjectRoot')
CLASSIFICATION_PREP_DIR = DRIVE_ROOT / "results" / "classification_prep"
CHECKPOINTS_DIR = DRIVE_ROOT / "checkpoints"
RUN_REGISTRY_PATH = resolve_run_registry_path(drive_root=DRIVE_ROOT, repo_dir=REPO_DIR)

# Choose one tokenizer family and one exact pretrained experiment name.
# The notebook can either compare model variants at one fixed layer or hold
# one saved model fixed and compare several internal layers.
TOKENIZER_FAMILY = "glyberta"
EXPERIMENT_NAME = "mlm15_L6_H512_A8_lr00001_ep100_setv1_train_only"
COMPARISON_MODE = "layer_progression"
MODEL_VARIANTS = (
    "pretrained_mlm",
    "classification_mlm_init",
    "classification_random_init",
)
FIXED_EMBEDDING_LAYER_INDEX = 0

# Choose which label source should define the notebook-14 probe target.
# - CURRENT_PROJECT_LABEL_SOURCE keeps using the original notebook-09 labels
# - STRUCTURAL_RULE_LABEL_SOURCE reuses the structural exports from notebook 15
PROBE_LABEL_SOURCE = STRUCTURAL_RULE_LABEL_SOURCE
STRUCTURAL_RUN_LABEL = "compact_iupac_structural_rules_v1"
# For the structural label source, the recommended main experiment excludes
# rows where the structural rules and provided labels still directly
# contradict each other.
STRUCTURAL_CONTRADICTION_POLICY = DEFAULT_STRUCTURAL_CONTRADICTION_POLICY

# Choose which probe target to study.
# - "n_glycan_binary": N-glycan versus other using the active label source
# - "n_glycan_subclass_multiclass": compare `High mannose`, `Complex`,
#   and `Hybrid` within the active N-glycan label source
PROBE_TARGET_MODE = "n_glycan_subclass_multiclass"
N_GLYCAN_SUBCLASS_CATEGORIES = (
    "High mannose",
    "Complex",
    "Hybrid",
)

# Layer-progression mode settings.
MODEL_VARIANT = "pretrained_mlm"
MODEL_SUBDIR = "best_model"
# When COMPARISON_MODE = "layer_progression", compare the input embeddings,
# one middle layer, and the final encoder layer for this default L6 model.
EMBEDDING_LAYER_INDICES = (
    0,
    3,
    -1,
)

# These candidate classifier labels only matter when MODEL_VARIANT is switched
# to one of the fine-tuned lineages later.
CLASSIFIER_MLM_RUN_LABEL_CANDIDATES = (
    "cls_lr2e-5_ep100_bs16_mlm",
    "cls_lr2e-5_ep10_bs16_mlm",
)
CLASSIFIER_RANDOM_RUN_LABEL_CANDIDATES = (
    "cls_lr2e-5_ep100_bs16_randominit",
    "cls_lr2e-5_ep10_bs16_randominit",
    "cls_lr2e-5_ep10_bs16_random_init",
)

# Embedding, UMAP, and probe settings.
POOLING_STRATEGY = "mean"
SPLITS_TO_INCLUDE = ("train", "val", "test")
# Keep the full corpus in the embedding view, but fit the logistic
# regression only on train and judge it only on held-out test.
TRAIN_SPLITS = ("train",)
EVALUATION_SPLITS = ("test",)
# This setting only affects the original notebook-09 binary labels. The
# structural-rule exports from notebook 15 already provide a full-corpus
# binary target, so this switch is ignored when PROBE_LABEL_SOURCE is
# STRUCTURAL_RULE_LABEL_SOURCE.
EXCLUDE_UNLABELED_ROWS = False
PROBABILITY_THRESHOLD = 0.5
LOGREG_C = 1.0
CLASS_WEIGHT = "balanced"
MAX_ITER = 2000
RANDOM_STATE = 42
BATCH_SIZE = 32
MAX_LENGTH = None
UMAP_NEIGHBORS = 15
UMAP_MIN_DIST = 0.10
UMAP_METRIC = "cosine"
UMAP_COLOR_COLUMN = "probe_target_label"
HIGHLIGHT_NEIGHBOR_COUNT = 10
HIGHLIGHT_ACCESSIONS = (
    "G60230HH",
    "G89864BN",
    "G27893KR",
    "G56734EJ",
)

# Output and overwrite behavior.
HTML_REPORT_TITLE = "N-glycan subclass layer progression comparison"
COMPARISON_RUN_LABEL = "n_glycan_subclass_pretrained_layer_comparison"
OVERWRITE_EXISTING_OUTPUTS = True
FAIL_ON_MISSING_MODEL_DIRS = False
METRIC_PLOT_NAMES = ("accuracy", "f1", "balanced_accuracy", "weighted_f1")

# Configure the optional clean public-export folder.
PUBLIC_EXPORT_ENABLED = False
PUBLIC_EXPORT_PARENT_SUBDIR = 'results/public_reports'
PUBLIC_GITHUB_OWNER = 'hb791-dev'
PUBLIC_GITHUB_REPO = 'glycan-roberta'
PUBLIC_GITHUB_REF = 'main'
PUBLIC_EXPORT_FAIL_ON_SENSITIVE_MATCH = True

SUPPORTED_COMPARISON_MODES = ("variant_fixed_layer", "layer_progression")
if COMPARISON_MODE not in SUPPORTED_COMPARISON_MODES:
    raise ValueError(
        f"Unsupported COMPARISON_MODE {COMPARISON_MODE!r}. "
        f"Choose from {SUPPORTED_COMPARISON_MODES}."
    )

if PROBE_LABEL_SOURCE not in SUPPORTED_PROBE_LABEL_SOURCES:
    raise ValueError(
        f"Unsupported PROBE_LABEL_SOURCE {PROBE_LABEL_SOURCE!r}. "
        f"Choose from {SUPPORTED_PROBE_LABEL_SOURCES}."
    )

if STRUCTURAL_CONTRADICTION_POLICY not in SUPPORTED_STRUCTURAL_CONTRADICTION_POLICIES:
    raise ValueError(
        f"Unsupported STRUCTURAL_CONTRADICTION_POLICY {STRUCTURAL_CONTRADICTION_POLICY!r}. "
        f"Choose from {SUPPORTED_STRUCTURAL_CONTRADICTION_POLICIES}."
    )

SUPPORTED_PROBE_TARGET_MODES = ("n_glycan_binary", "n_glycan_subclass_multiclass")
if PROBE_TARGET_MODE not in SUPPORTED_PROBE_TARGET_MODES:
    raise ValueError(
        f"Unsupported PROBE_TARGET_MODE {PROBE_TARGET_MODE!r}. "
        f"Choose from {SUPPORTED_PROBE_TARGET_MODES}."
    )

validate_tokenizer_family(TOKENIZER_FAMILY, supported_families=SUPPORTED_TOKENIZER_FAMILIES)
print(f"Drive root: {DRIVE_ROOT}")
print(f"Classification prep dir: {CLASSIFICATION_PREP_DIR}")
print(f"Checkpoints dir: {CHECKPOINTS_DIR}")
print(f"Run registry path: {RUN_REGISTRY_PATH}")
print(f"Tokenizer family: {TOKENIZER_FAMILY}")
print(f"Experiment name: {EXPERIMENT_NAME}")
print(f"Comparison mode: {COMPARISON_MODE}")
print(f"Probe label source: {PROBE_LABEL_SOURCE}")
print(f"Structural run label: {STRUCTURAL_RUN_LABEL}")
print(f"Structural contradiction policy: {STRUCTURAL_CONTRADICTION_POLICY}")
print(f"Model variants: {MODEL_VARIANTS}")
print(f"Fixed embedding layer index: {FIXED_EMBEDDING_LAYER_INDEX}")
print(f"Probe target mode: {PROBE_TARGET_MODE}")
print(f"N-glycan subclass categories: {N_GLYCAN_SUBCLASS_CATEGORIES}")
print(f"Model variant: {MODEL_VARIANT}")
print(f"Model subdir: {MODEL_SUBDIR}")
print(f"Embedding layer indices: {EMBEDDING_LAYER_INDICES}")
print(f"Classifier, MLM init candidates: {CLASSIFIER_MLM_RUN_LABEL_CANDIDATES}")
print(f"Classifier, random-init candidates: {CLASSIFIER_RANDOM_RUN_LABEL_CANDIDATES}")
print(f"Pooling strategy: {POOLING_STRATEGY}")
print(f"Keep unlabeled rows from original labels: {not EXCLUDE_UNLABELED_ROWS}")
print(f"UMAP color column: {UMAP_COLOR_COLUMN}")
print(f"Highlight accessions: {HIGHLIGHT_ACCESSIONS}")
print(f"Comparison run label: {COMPARISON_RUN_LABEL}")


## Build the requested comparison from the cleaned registry

This step finds one exact experiment in the cleaned registry, resolves the saved lineage that the notebook should follow, expands either the ordered layer list or the ordered model-variant list into run specs, and records the exact saved model folders that the probe will use.

**Expected output**
- a compact manifest of the compared model states
- a count of how many requested model directories currently exist in Drive
- one saved config JSON in the notebook-14 output folder

**How to interpret the result**
- if the cell raises an error about multiple matched experiments, narrow `EXPERIMENT_NAME` until the manifest represents one architecture only
- the classifier run-label resolution table only matters when one of the compared variants is a fine-tuned classifier
- the manifest is the main place to confirm that tokenizer family, architecture label, compared variants or layers, and saved model folders match the intended comparison
- the printed public-export paths show where the clean shareable HTML folder will be copied if that option is enabled


In [ ]:
# Load the cleaned registry so the selected comparison mode can reuse the saved
# run metadata instead of hardcoding architecture details.
run_registry_df = load_run_registry(RUN_REGISTRY_PATH)
classifier_label_resolution = resolve_classifier_run_label_choices(
    checkpoints_dir=CHECKPOINTS_DIR,
    tokenizer_family=TOKENIZER_FAMILY,
    experiment_name=EXPERIMENT_NAME,
    classifier_mlm_run_label_candidates=CLASSIFIER_MLM_RUN_LABEL_CANDIDATES,
    classifier_random_run_label_candidates=CLASSIFIER_RANDOM_RUN_LABEL_CANDIDATES,
    model_subdir=MODEL_SUBDIR,
)
classifier_label_resolution_df = build_classifier_run_label_resolution_table(
    classifier_label_resolution,
)
classifier_mlm_run_label = classifier_label_resolution["classification_mlm_init"]["selected_run_label"]
classifier_random_run_label = classifier_label_resolution["classification_random_init"]["selected_run_label"]

if COMPARISON_MODE == "variant_fixed_layer":
    run_specs = build_registry_run_specs(
        run_registry_df,
        tokenizer_families=[TOKENIZER_FAMILY],
        experiment_names=[EXPERIMENT_NAME],
        model_variants=MODEL_VARIANTS,
        classifier_mlm_run_label=classifier_mlm_run_label,
        classifier_random_run_label=classifier_random_run_label,
    )
else:
    run_specs = build_layer_run_specs(
        run_registry_df,
        checkpoints_dir=CHECKPOINTS_DIR,
        tokenizer_family=TOKENIZER_FAMILY,
        experiment_name=EXPERIMENT_NAME,
        model_variant=MODEL_VARIANT,
        model_subdir=MODEL_SUBDIR,
        embedding_layer_indices=EMBEDDING_LAYER_INDICES,
        classifier_mlm_run_label=classifier_mlm_run_label,
        classifier_random_run_label=classifier_random_run_label,
    )

if not run_specs:
    raise ValueError("No run specs were generated. Adjust TOKENIZER_FAMILY or EXPERIMENT_NAME.")

# Build the notebook-14 output folder and one manifest row per requested model state.
output_paths = build_embedding_logreg_output_paths(
    DRIVE_ROOT,
    tokenizer_family=TOKENIZER_FAMILY,
    experiment_name=EXPERIMENT_NAME,
    comparison_run_label=COMPARISON_RUN_LABEL,
)
PUBLIC_REPORT_NOTEBOOK_STEM = '14_n_glycan_logistic_regression'
PUBLIC_EXPORT_PATH_PARTS = [TOKENIZER_FAMILY, EXPERIMENT_NAME, COMPARISON_RUN_LABEL]
PUBLIC_EXPORT_PARENT_DIR = DRIVE_ROOT / PUBLIC_EXPORT_PARENT_SUBDIR
PUBLIC_EXPORT_REPO_SUBDIR = build_public_report_subdir(
    PUBLIC_REPORT_NOTEBOOK_STEM,
    PUBLIC_EXPORT_PATH_PARTS,
)
PUBLIC_EXPORT_DIR = build_public_export_dir(
    PUBLIC_EXPORT_PARENT_DIR,
    PUBLIC_REPORT_NOTEBOOK_STEM,
    PUBLIC_EXPORT_PATH_PARTS,
)
run_manifest_df = build_run_manifest(run_specs, checkpoints_dir=CHECKPOINTS_DIR, model_subdir=MODEL_SUBDIR)

# Save the active notebook settings so the exact comparison can be audited
# later without reopening the notebook itself.
run_config = build_embedding_logreg_run_config(
    drive_root=DRIVE_ROOT,
    classification_prep_dir=CLASSIFICATION_PREP_DIR,
    checkpoints_dir=CHECKPOINTS_DIR,
    pooling_strategy=POOLING_STRATEGY,
    embedding_layer_index=(FIXED_EMBEDDING_LAYER_INDEX if COMPARISON_MODE == "variant_fixed_layer" else None),
    embedding_layer_indices=(None if COMPARISON_MODE == "variant_fixed_layer" else EMBEDDING_LAYER_INDICES),
    splits_to_include=SPLITS_TO_INCLUDE,
    train_splits=TRAIN_SPLITS,
    evaluation_splits=EVALUATION_SPLITS,
    exclude_unlabeled_rows=EXCLUDE_UNLABELED_ROWS,
    probability_threshold=PROBABILITY_THRESHOLD,
    logreg_c=LOGREG_C,
    class_weight=CLASS_WEIGHT,
    max_iter=MAX_ITER,
    random_state=RANDOM_STATE,
    batch_size=BATCH_SIZE,
    max_length=MAX_LENGTH,
    comparison_run_label=COMPARISON_RUN_LABEL,
    output_paths=output_paths,
    run_specs=run_specs,
)
run_config['probe_target'] = {
    'probe_label_source': PROBE_LABEL_SOURCE,
    'structural_run_label': (STRUCTURAL_RUN_LABEL if PROBE_LABEL_SOURCE == STRUCTURAL_RULE_LABEL_SOURCE else None),
    'structural_contradiction_policy': (
        STRUCTURAL_CONTRADICTION_POLICY
        if PROBE_LABEL_SOURCE == STRUCTURAL_RULE_LABEL_SOURCE
        else None
    ),
    'probe_target_mode': PROBE_TARGET_MODE,
    'exclude_unlabeled_rows': bool(EXCLUDE_UNLABELED_ROWS),
    'n_glycan_subclass_categories': list(N_GLYCAN_SUBCLASS_CATEGORIES),
    'umap_color_column': UMAP_COLOR_COLUMN,
}
if COMPARISON_MODE == "variant_fixed_layer":
    run_config['variant_fixed_layer_comparison'] = {
        'model_variants': list(MODEL_VARIANTS),
        'model_subdir': MODEL_SUBDIR,
        'fixed_embedding_layer_index': int(FIXED_EMBEDDING_LAYER_INDEX),
        'highlight_accessions': list(HIGHLIGHT_ACCESSIONS),
        'highlight_neighbor_count': HIGHLIGHT_NEIGHBOR_COUNT,
        'umap_neighbors': UMAP_NEIGHBORS,
        'umap_min_dist': UMAP_MIN_DIST,
        'umap_metric': UMAP_METRIC,
        'umap_color_column': UMAP_COLOR_COLUMN,
    }
else:
    run_config['layer_comparison'] = {
        'model_variant': MODEL_VARIANT,
        'model_subdir': MODEL_SUBDIR,
        'embedding_layer_indices': list(EMBEDDING_LAYER_INDICES),
        'highlight_accessions': list(HIGHLIGHT_ACCESSIONS),
        'highlight_neighbor_count': HIGHLIGHT_NEIGHBOR_COUNT,
        'umap_neighbors': UMAP_NEIGHBORS,
        'umap_min_dist': UMAP_MIN_DIST,
        'umap_metric': UMAP_METRIC,
        'umap_color_column': UMAP_COLOR_COLUMN,
    }
run_config['public_export'] = {
    'enabled': bool(PUBLIC_EXPORT_ENABLED),
    'public_export_parent_subdir': PUBLIC_EXPORT_PARENT_SUBDIR,
    'public_export_dir': str(PUBLIC_EXPORT_DIR),
    'public_export_repo_subdir': PUBLIC_EXPORT_REPO_SUBDIR,
    'public_github_owner': PUBLIC_GITHUB_OWNER,
    'public_github_repo': PUBLIC_GITHUB_REPO,
    'public_github_ref': PUBLIC_GITHUB_REF,
    'public_export_fail_on_sensitive_match': bool(PUBLIC_EXPORT_FAIL_ON_SENSITIVE_MATCH),
}
run_config['classifier_run_label_resolution'] = classifier_label_resolution_df.to_dict(orient='records')
save_embedding_logreg_run_config(output_paths["run_config_path"], run_config)

if COMPARISON_MODE == "variant_fixed_layer":
    display_columns = [
        "display_label",
        "tokenizer_family",
        "architecture_label",
        "model_variant",
        "model_dir_exists",
    ]
else:
    display_columns = [
        "display_label",
        "tokenizer_family",
        "architecture_label",
        "model_variant",
        "model_subdir",
        "layer_label",
        "embedding_layer_index",
        "model_dir_exists",
    ]
print("Classifier run-label resolution")
display(classifier_label_resolution_df)
print("Requested comparison manifest")
display(run_manifest_df[display_columns])
print(f"Requested runs: {len(run_manifest_df)}")
print(f"Existing model dirs: {int(run_manifest_df['model_dir_exists'].sum())}")
print(f"Output dir: {output_paths['results_dir']}")
print(f"HTML report path: {output_paths['html_report_path']}")
print(f"Public export Drive folder: {PUBLIC_EXPORT_DIR}")
print(f"Repo destination after copy: {PUBLIC_EXPORT_REPO_SUBDIR}")


## Load the chosen label source and build the requested probe target

This step prepares the notebook-14 target table from whichever label source you selected near the top of the notebook. The workflow now supports two defensible sources:

- `CURRENT_PROJECT_LABEL_SOURCE`: reuse the original notebook-09 labels, including their known gaps and edge cases
- `STRUCTURAL_RULE_LABEL_SOURCE`: reuse the compact-IUPAC structural exports written by notebook 15 so the probe can cover many more glycans without silently relabeling rows inside notebook 14

When the structural source is active, `STRUCTURAL_CONTRADICTION_POLICY` decides whether rows marked as `true_contradiction` in notebook 15 stay in the probe dataset or are excluded from the main comparison. The recommended layer-comparison setting is `exclude_true_contradictions`.

The target definitions stay the same after the source is chosen:

- `n_glycan_binary`: positive class = `N-glycan`, negative class = everything else in the chosen source
- `n_glycan_subclass_multiclass`: compare `High mannose`, `Complex`, and `Hybrid` within the chosen N-glycan source

The split logic here is intentionally strict for the probe: fit the logistic regression on `train`, then judge the compared embedding layers only on the held-out `test` split. The shared UMAP later reuses the full filtered corpus so the geometry view can include the full selected label source across `train`, `val`, and `test`.

**Expected output**
- the active probe label source and target name
- the number of rows kept for the selected probe
- a split-by-target count table for the active binary or multiclass view
- a broader class count table for a quick sanity check
- edge-case summary tables that document unresolved structural rows, excluded structural subclasses, excluded contradiction rows, or reference-only label mismatches

**How to interpret the result**
- if the structural source keeps far more rows than the original labels, that is expected and reflects the broader compact-IUPAC coverage from notebook 15
- excluding `true_contradiction` rows is helpful for the main layer experiment because it keeps parser-versus-label conflicts from muddying the representation comparison
- for the structural subclass task, `paucimannose_or_truncated` and `unresolved` rows should stay documented in the edge-case tables rather than being forced into the clean 3-class probe
- if the broader class counts look surprising, inspect the saved structural exports or notebook-09 input tables before trusting the downstream probe metrics


In [ ]:
# Load the requested notebook-14 probe target from either the original
# notebook-09 labels or the structural exports written by notebook 15.
probe_bundle = prepare_probe_dataframe_for_notebook14(
    probe_label_source=PROBE_LABEL_SOURCE,
    train_csv_path=CLASSIFICATION_PREP_DIR / "train_classification.csv",
    val_csv_path=CLASSIFICATION_PREP_DIR / "val_classification.csv",
    test_csv_path=CLASSIFICATION_PREP_DIR / "test_classification.csv",
    label_vocabulary_path=CLASSIFICATION_PREP_DIR / "label_vocabulary.csv",
    splits_to_include=SPLITS_TO_INCLUDE,
    probe_target_mode=PROBE_TARGET_MODE,
    exclude_unlabeled_rows=EXCLUDE_UNLABELED_ROWS,
    subclass_categories=N_GLYCAN_SUBCLASS_CATEGORIES,
    structural_run_label=STRUCTURAL_RUN_LABEL,
    structural_contradiction_policy=STRUCTURAL_CONTRADICTION_POLICY,
    project_root=DRIVE_ROOT,
)
annotated_probe_df = probe_bundle['annotated_probe_df']
label_vocabulary_df = probe_bundle['label_vocabulary_df']
target_summary_df = probe_bundle['target_summary_df']
class_summary_df = probe_bundle['class_summary_df']
edge_case_summary_df = probe_bundle['edge_case_summary_df']
edge_case_detail_df = probe_bundle['edge_case_detail_df']

# Surface the active source and any key caveats before the notebook starts
# embedding and fitting probes.
print(f"Probe label source: {probe_bundle['probe_label_source']}")
print(f"Rows kept for the probe: {len(annotated_probe_df)}")
print(f"Probe target name: {probe_bundle['probe_target_name']}")
if PROBE_LABEL_SOURCE == STRUCTURAL_RULE_LABEL_SOURCE:
    structural_output_paths = classification_embedding_logreg.build_structural_classification_output_paths(
        DRIVE_ROOT,
        run_label=STRUCTURAL_RUN_LABEL,
    )
    print(f"Structural results dir: {structural_output_paths['results_dir']}")
    print(f"Structural contradiction policy: {probe_bundle['structural_contradiction_policy']}")
elif PROBE_TARGET_MODE == 'n_glycan_binary' and EXCLUDE_UNLABELED_ROWS:
    print('Binary probe excludes unlabeled rows from the original label source.')

print("Probe target counts by split")
display(target_summary_df)
print("Broad class counts used by this probe")
display(class_summary_df)
print("Probe edge-case summary")
display(edge_case_summary_df)
if not edge_case_detail_df.empty:
    print("Probe edge-case detail preview")
    display(edge_case_detail_df.head(40))


## Run the fixed-layer or layer-progression probe suite

For each requested model state, the notebook will:

- resolve the saved model folder for that run
- embed the glycan sequences using either the fixed layer or the per-run layer setting
- fit one logistic regression on the train split only
- evaluate that same probe on the held-out test split
- when layer mode is selected, build a shared UMAP across the requested layers using the same filtered corpus
- when layer mode is selected, highlight the requested glycan accessions across every layer panel
- save per-run predictions, metric summary grids, logistic-regression diagnostics, and one comparison HTML report
- optionally copy a clean public-ready HTML folder for later GitHub publishing

**Expected output**
- a count of completed compared runs
- a count of skipped runs whose model folders were missing
- the main saved output paths for the metrics table, test summary, and HTML report
- saved ROC, precision-recall, confusion-matrix, and probability-distribution plots for every mode
- in layer mode, saved layer-progression and shared-UMAP plots
- when public export is enabled, copied-file and scan tables for the clean public folder

**How to interpret the result**
- this step is the longest part of the notebook because it recomputes embeddings for each requested run
- skipped runs are not automatically fatal unless `FAIL_ON_MISSING_MODEL_DIRS` is set to `True`
- if the step fails during embedding, check the saved model folders, selected variants, and requested layer settings first


In [ ]:
# Run the full notebook-14 workflow for the selected comparison mode.
build_layer_progression = COMPARISON_MODE == "layer_progression"
fixed_embedding_layer_index = FIXED_EMBEDDING_LAYER_INDEX if COMPARISON_MODE == "variant_fixed_layer" else None

probe_results = run_embedding_logreg_suite(
    annotated_df=annotated_probe_df,
    run_specs=run_specs,
    checkpoints_dir=CHECKPOINTS_DIR,
    output_paths=output_paths,
    target_summary_df=target_summary_df,
    edge_case_summary_df=edge_case_summary_df,
    edge_case_detail_df=edge_case_detail_df,
    pooling_strategy=POOLING_STRATEGY,
    embedding_layer_index=fixed_embedding_layer_index,
    batch_size=BATCH_SIZE,
    max_length=MAX_LENGTH,
    train_splits=TRAIN_SPLITS,
    evaluation_splits=EVALUATION_SPLITS,
    probability_threshold=PROBABILITY_THRESHOLD,
    regularization_c=LOGREG_C,
    class_weight=CLASS_WEIGHT,
    max_iter=MAX_ITER,
    random_state=RANDOM_STATE,
    overwrite_existing_outputs=OVERWRITE_EXISTING_OUTPUTS,
    fail_on_missing_model_dirs=FAIL_ON_MISSING_MODEL_DIRS,
    metric_plot_names=METRIC_PLOT_NAMES,
    report_title=HTML_REPORT_TITLE,
    build_snapshot_progression=build_layer_progression,
    highlight_accessions=HIGHLIGHT_ACCESSIONS,
    highlight_neighbor_count=HIGHLIGHT_NEIGHBOR_COUNT,
    umap_neighbors=UMAP_NEIGHBORS,
    umap_min_dist=UMAP_MIN_DIST,
    umap_metric=UMAP_METRIC,
    umap_random_state=RANDOM_STATE,
    umap_color_column=UMAP_COLOR_COLUMN,
)

print(f"Completed runs: {probe_results['metrics_df']['display_label'].nunique()}")
print(f"Skipped runs: {len(probe_results['skipped_df'])}")
print(f"Saved metrics table: {output_paths['split_metrics_path']}")
print(f"Saved test summary: {output_paths['test_summary_path']}")
print(f"Saved edge-case summary: {output_paths['edge_case_summary_path']}")
print(f"Saved edge-case detail table: {output_paths['edge_case_detail_path']}")
snapshot_analysis = probe_results['snapshot_analysis']
if build_layer_progression and snapshot_analysis.get('snapshot_progression_plot_path') is not None:
    print(f"Saved layer metric progression: {snapshot_analysis['snapshot_progression_plot_path']}")
if build_layer_progression and snapshot_analysis.get('snapshot_umap_plot_path') is not None:
    print(f"Saved shared layer UMAP grid: {snapshot_analysis['snapshot_umap_plot_path']}")
if build_layer_progression:
    print(f"Saved highlighted accession table: {output_paths['highlight_accession_table_path']}")
    print(f"Saved highlighted similarity table: {output_paths['highlight_similarity_table_path']}")
    print(f"Saved highlighted neighbor table: {output_paths['highlight_neighbor_table_path']}")
for plot_name, plot_path in probe_results["diagnostic_plot_paths"].items():
    if plot_path is not None:
        print(f"Saved diagnostic plot [{plot_name}]: {plot_path}")
print(f"HTML report: {probe_results['html_report_path']}")

display(HTML(
    f'<p><a href="{probe_results["html_report_path"]}" target="_blank">Open full HTML logistic-regression report</a></p>'
))

public_export_artifacts = None

if PUBLIC_EXPORT_ENABLED:
    public_export_artifacts = export_public_embedding_logreg_html(
        probe_results=probe_results,
        export_dir=PUBLIC_EXPORT_DIR,
        repo_public_subdir=PUBLIC_EXPORT_REPO_SUBDIR,
        repo_owner=PUBLIC_GITHUB_OWNER,
        repo_name=PUBLIC_GITHUB_REPO,
        repo_ref=PUBLIC_GITHUB_REF,
    )

    print(f'Public export Drive folder: {public_export_artifacts["public_export_dir"]}')
    print(f'Repo folder to copy into before push: {PUBLIC_EXPORT_REPO_SUBDIR}')
    print(f'Repo report path after push: {public_export_artifacts["repo_index_path"]}')
    print(f'GitHack URL after push: {public_export_artifacts["githack_url"]}')
    print('')

    print('=== Copied public files ===')
    display(public_export_artifacts['copied_files_df'])

    print('=== Dependency issues ===')
    if public_export_artifacts['dependency_issues_df'].empty:
        print('No missing local HTML dependencies were found in the public export.')
    else:
        display(public_export_artifacts['dependency_issues_df'])

    print('=== Sensitive-string scan ===')
    if public_export_artifacts['scan_results_df'].empty:
        print('No obvious local Drive paths were found in the copied files.')
    else:
        display(public_export_artifacts['scan_results_df'])

    if public_export_artifacts['has_dependency_issues']:
        raise ValueError(
            'The public export still has missing local dependencies. Fix those before any GitHub copy step.'
        )

    if public_export_artifacts['has_sensitive_matches'] and PUBLIC_EXPORT_FAIL_ON_SENSITIVE_MATCH:
        raise ValueError(
            'The public export still contains suspicious local-environment strings. Review the scan table before sharing the files.'
        )
else:
    print('PUBLIC_EXPORT_ENABLED is False, so the notebook skipped the clean public-export step.')


## Review the held-out test summary and saved report

This notebook is meant to judge either a fixed-layer variant comparison or a layer-progression comparison on the held-out test split. The inline review therefore keeps the held-out test metrics central and adds shared-UMAP outputs only when layer mode is selected.

**Expected output**
- the held-out test summary table for the compared runs
- the saved held-out test metric-grid image when that split has results
- held-out test ROC, precision-recall, confusion-matrix, and probability-distribution plots
- in layer mode, the saved layer-progression metric plot and shared-UMAP grid
- in layer mode, compact tables for the highlighted accession positions and pairwise similarities
- an optional skipped-run table when the requested model folder was missing
- a saved HTML report link from the previous step for easier side-by-side review outside the notebook
- when enabled, public-export copy and scan tables that help verify the report is safe to share

**How to interpret the result**
- use the held-out test table as the main quantitative comparison view
- if fixed-layer variant mode suggests the pretrained input layer is already strong, compare that table against the random-init and MLM-init variants before deciding whether the layer-0 signal is really worth following up
- if layer mode shows two layers are close on the test metrics, inspect the shared UMAP and highlighted-neighbor tables before treating them as interchangeable


In [ ]:
# Keep the inline review focused on a compact set of metrics that are easy to
# compare across whichever run mode this notebook used. Some columns are only
# present for snapshot or layer progression runs, so filter the preferred list
# down to the columns that actually exist in the returned test summary table.
preferred_summary_columns = [
    "display_label",
    "tokenizer_family",
    "architecture_label",
    "model_variant",
    "target_kind",
    "target_class_count",
    "snapshot_label",
    "layer_label",
    "embedding_layer_index",
    "row_count",
    "positive_count",
    "precision",
    "recall",
    "roc_auc",
    "average_precision",
    "f1",
    "macro_f1",
    "weighted_f1",
    "balanced_accuracy",
    "accuracy",
]
available_summary_columns = [
    column_name
    for column_name in preferred_summary_columns
    if column_name in probe_results["test_summary_df"].columns
]

print("Test summary")
display(probe_results["test_summary_df"][available_summary_columns])

if not probe_results["edge_case_summary_df"].empty:
    print("Probe edge-case summary")
    display(probe_results["edge_case_summary_df"])

snapshot_progression_plot_path = probe_results['snapshot_analysis'].get('snapshot_progression_plot_path')
if snapshot_progression_plot_path:
    print("Snapshot metric progression")
    display(HTML(build_responsive_image_html(snapshot_progression_plot_path, alt_text="Snapshot metric progression")))

snapshot_umap_plot_path = probe_results['snapshot_analysis'].get('snapshot_umap_plot_path')
if snapshot_umap_plot_path:
    print("Shared UMAP with highlighted glycans")
    display(HTML(build_responsive_image_html(snapshot_umap_plot_path, alt_text="Snapshot UMAP grid")))

test_metric_grid_path = probe_results["plot_paths"].get("test")
if test_metric_grid_path:
    print("Test metric grid")
    display(HTML(build_responsive_image_html(test_metric_grid_path, alt_text="Test metric grid")))

diagnostic_plot_order = [
    ("test_roc", "Test ROC curve"),
    ("test_pr", "Test precision-recall curve"),
    ("test_confusion", "Test confusion matrices"),
    ("test_probability", "Test probability distributions"),
]

for plot_key, plot_title in diagnostic_plot_order:
    plot_path = probe_results["diagnostic_plot_paths"].get(plot_key)
    if plot_path:
        print(plot_title)
        display(HTML(build_responsive_image_html(plot_path, alt_text=plot_title)))

highlight_positions_df = probe_results['snapshot_analysis'].get('highlight_positions_df', pd.DataFrame())
if not highlight_positions_df.empty:
    print("Highlighted accession positions")
    display(highlight_positions_df)

highlight_similarity_df = probe_results['snapshot_analysis'].get('highlight_similarity_df', pd.DataFrame())
if not highlight_similarity_df.empty:
    print("Highlighted accession pairwise similarity")
    display(highlight_similarity_df)

highlight_neighbors_df = probe_results['snapshot_analysis'].get('highlight_neighbors_df', pd.DataFrame())
if not highlight_neighbors_df.empty:
    print("Highlighted accession nearest neighbors")
    display(highlight_neighbors_df.head(40))

if not probe_results["skipped_df"].empty:
    print("Skipped snapshots")
    display(probe_results["skipped_df"])
